# Session 4: Group Project Kickoff - DB2, Polars, and Streamlit

In sessions 4 and 5 you will build a group analytics project with a real database:

1. Connect to the `ATTPLANE` DB2 database.
2. Discover the tables available to your group.
3. Read database tables into Polars.
4. Clean and prepare analytical datasets.
5. Build a Streamlit dashboard that communicates useful insights.

The goal is not only to make charts. The goal is to build a small analytics product: reliable data loading, clear transformations, useful metrics, and an interface that helps someone explore the aviation data.

## Session Agenda

1. Environment and DB2 connection test
2. Table discovery and safe table reads with Polars
3. Initial data profiling
4. Building reusable Polars analytics functions
5. Exporting prepared data for a faster dashboard
6. Creating a simple Streamlit dashboard
7. Group assignment checklist

## 0. Setup

The database connection uses these packages:

- `sqlalchemy`: generic database connection layer
- `ibm_db` and `ibm_db_sa`: IBM DB2 driver and SQLAlchemy dialect
- `polars`: data processing
- `plotly`: charts
- `streamlit`: dashboard app

If your environment is missing any of them, install/update the project environment from a terminal:

```bash
uv sync
```

If DB2 driver installation is slow on your machine, first verify that the database port is reachable with the connectivity cell below. A reachable port means networking is OK; a slow install is usually the native `ibm_db` package setup.

In [ ]:
import os
import socket
from pathlib import Path
from urllib.parse import quote_plus

import polars as pl
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

print(f"Polars version: {pl.__version__}")

### Pandas Comparison

| Task | Pandas-first habit | Polars-first habit |
|---|---|---|
| Load tabular data | `pd.read_sql(query, conn)` | `pl.read_database(query, connection=conn)` |
| Inspect schema | `df.dtypes` | `df.schema` |
| Transform columns | Mutate with `df["col"] = ...` | Build expressions with `.with_columns(...)` |
| Aggregate | `df.groupby(...).agg(...)` | `df.group_by(...).agg(...)` |
| Optimize large pipelines | Usually manual | Use `.lazy()` so Polars can optimize |

## 1. Database Connection Details

The class database is the DB2 database shown in DBeaver:

- Host: `52.211.123.34`
- Port: `25010`
- Database: `ATTPLANE`
- User: `attgrp1` through `attgrp8`
- Password: `bigdata`

This notebook defaults to `attgrp6` and `bigdata` so the first connection test is concrete. For your group project, change the user to your assigned group account.

For a cleaner local setup, create a `.env` file in the repository root:

```text
DB_HOST=52.211.123.34
DB_PORT=25010
DB_NAME=ATTPLANE
DB_USERNAME=attgrp6
DB_PASSWORD=bigdata
```

Do not commit personal or private passwords. The `bigdata` password is a shared teaching credential for this course database.

In [ ]:
DB_HOST = os.getenv("DB_HOST", "52.211.123.34")
DB_PORT = int(os.getenv("DB_PORT", "25010"))
DB_NAME = os.getenv("DB_NAME", "ATTPLANE")
DB_USERNAME = os.getenv("DB_USERNAME", "attgrp6")
DB_PASSWORD = os.getenv("DB_PASSWORD", "bigdata")
DB_SCHEMA = os.getenv("DB_SCHEMA", DB_USERNAME.upper())

print({
    "host": DB_HOST,
    "port": DB_PORT,
    "database": DB_NAME,
    "username": DB_USERNAME,
    "schema": DB_SCHEMA,
    "password": "***",
})

### 1.1 Quick Network Check

This only checks whether your computer can reach the DB2 server port. It does **not** authenticate and it does **not** prove that the DB2 Python driver is installed.

Use this when diagnosing problems:

- If this fails, you likely have a networking/VPN/firewall issue.
- If this succeeds but SQLAlchemy fails to import or connect, focus on Python packages, credentials, or DB2 driver setup.

In [ ]:
def tcp_check(host: str = DB_HOST, port: int = DB_PORT, timeout: float = 8.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        sock.connect((host, port))
    return True

try:
    tcp_check()
    print(f"TCP connection succeeded: {DB_HOST}:{DB_PORT} is reachable")
except Exception as exc:
    print(f"TCP connection failed: {type(exc).__name__}: {exc}")

### 1.2 Create the SQLAlchemy Engine

The connection URL follows the DB2 SQLAlchemy dialect pattern:

```text
db2+ibm_db://username:password@host:port/database
```

We URL-encode the username and password so special characters do not break the connection string.

In [ ]:
def make_db2_engine():
    user = quote_plus(DB_USERNAME)
    password = quote_plus(DB_PASSWORD)
    url = f"db2+ibm_db://{user}:{password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    return create_engine(url, pool_pre_ping=True)

engine = make_db2_engine()
engine

### 1.3 Test Authentication with `attgrp6` / `bigdata`

DB2 does not use `SELECT 1` without a table in the same way as some other databases. The standard one-row DB2 table is `SYSIBM.SYSDUMMY1`.

In [ ]:
def test_db_connection(engine) -> bool:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 AS ok FROM SYSIBM.SYSDUMMY1"))
        row = result.fetchone()
    return row is not None and row[0] == 1

try:
    assert test_db_connection(engine)
    print("DB2 connection successful")
except Exception as exc:
    print("DB2 connection failed")
    print(f"{type(exc).__name__}: {exc}")
    print("\nChecklist:")
    print("1. Did the TCP check succeed?")
    print("2. Are ibm_db and ibm_db_sa installed in this notebook kernel?")
    print("3. Are DB_USERNAME and DB_PASSWORD correct for your group?")
    print("4. Is DB_NAME exactly ATTPLANE?")

## 2. Discover the Database

Before reading a full table, discover what you have access to. Real databases often have many schemas and system tables. We will use DB2 catalog views to list the useful tables.

Important DB2 vocabulary:

- **Database**: `ATTPLANE`, the DB you connect to.
- **Schema**: a namespace inside the database. Tables are referenced as `SCHEMA.TABLE_NAME`.
- **Table**: the data object you read.

The DBeaver screenshot shows the database name, not necessarily the schema name. The cells below help you identify the schema and table names available to your account.

In [ ]:
def normalize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    """Make DB2 result columns easier to use in Python code."""
    return df.rename({col: col.strip().lower() for col in df.columns})


def read_sql_polars(query: str, params: dict | None = None) -> pl.DataFrame:
    """Run a SQL query and return a Polars DataFrame with normalized column names."""
    with engine.connect() as conn:
        df = pl.read_database(query=query, connection=conn, execute_options={"parameters": params or {}})
    return normalize_column_names(df)

current_context = read_sql_polars("""
SELECT
    CURRENT SERVER AS current_server,
    CURRENT SCHEMA AS current_schema,
    CURRENT USER AS current_user
FROM SYSIBM.SYSDUMMY1
""")

current_context

In [ ]:
tables = read_sql_polars("""
SELECT
    RTRIM(tabschema) AS schema_name,
    RTRIM(tabname) AS table_name,
    type AS table_type
FROM syscat.tables
WHERE type IN ('T', 'V')
  AND tabschema NOT LIKE 'SYS%'
  AND tabschema NOT IN ('NULLID', 'SQLJ')
ORDER BY tabschema, tabname
""")

tables

In [ ]:
print(f"Accessible tables/views: {tables.height}")

tables.group_by("schema_name", "table_type").agg(
    pl.len().alias("n_objects")
).sort("schema_name", "table_type")

### Pandas Comparison

```python
# Pandas
pd.read_sql("SELECT * FROM schema.table FETCH FIRST 100 ROWS ONLY", conn)

# Polars
pl.read_database("SELECT * FROM schema.table FETCH FIRST 100 ROWS ONLY", connection=conn)
```

The SQL is the same. The difference is the DataFrame library that receives the result.

## 3. Read Tables Safely with Polars

Do **not** start by reading every row from every table. First inspect a small sample, check row counts, and decide which tables are relevant.

The helper below quotes DB2 identifiers safely and limits the number of rows for exploration.

In [ ]:
def qident(identifier: str) -> str:
    """Quote a DB2 identifier such as a schema or table name."""
    return '"' + identifier.replace('"', '""') + '"'


def qualified_table(schema: str, table: str) -> str:
    return f"{qident(schema)}.{qident(table)}"


def preview_table(schema: str, table: str, limit: int = 10) -> pl.DataFrame:
    query = f"SELECT * FROM {qualified_table(schema, table)} FETCH FIRST {int(limit)} ROWS ONLY"
    return read_sql_polars(query)


def count_rows(schema: str, table: str) -> int:
    query = f"SELECT COUNT(*) AS n_rows FROM {qualified_table(schema, table)}"
    return read_sql_polars(query).item(0, "n_rows")

In [ ]:
# Choose a table to inspect from the assigned group schema.
if tables.height == 0:
    raise ValueError("No accessible user tables found. Check credentials or ask the instructor.")

SCHEMA_NAME = DB_SCHEMA
TABLE_NAME = "AIRPLANES"

print(f"Selected table: {SCHEMA_NAME}.{TABLE_NAME}")
print(f"Rows: {count_rows(SCHEMA_NAME, TABLE_NAME):,}")

sample_df = preview_table(SCHEMA_NAME, TABLE_NAME, limit=10)
sample_df

In [ ]:
sample_df.schema

In [ ]:
sample_df.describe()

### Read a Full Table

Once you know which table you need, read it into Polars.

For very large tables, avoid `SELECT *`. Select only the columns needed for your dashboard, or add a SQL `WHERE` clause.

In [ ]:
def read_table(schema: str, table: str, columns: list[str] | None = None, where: str | None = None) -> pl.DataFrame:
    if columns:
        select_clause = ", ".join(qident(col) for col in columns)
    else:
        select_clause = "*"

    query = f"SELECT {select_clause} FROM {qualified_table(schema, table)}"
    if where:
        query += f" WHERE {where}"

    return read_sql_polars(query)

# For class exploration, keep this commented until you choose the right table.
# df = read_table(SCHEMA_NAME, TABLE_NAME)
# df.shape

## 4. Profile and Prepare a Dataset

The exact columns depend on the Plane DB tables. The profiling functions below work with any table and help your group decide what can become an insight.

In [ ]:
def profile_dataframe(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "nulls": [df[col].null_count() for col in df.columns],
        "n_unique": [df[col].n_unique() for col in df.columns],
    }).with_columns(
        (pl.col("nulls") / max(df.height, 1)).alias("null_rate")
    ).sort("null_rate", descending=True)

# Use sample_df first. Replace with your full df after loading.
profile_dataframe(sample_df)

In [ ]:
def standardize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    """Convert database column names to lowercase names for Python analysis."""
    rename_map = {col: col.strip().lower().replace(" ", "_") for col in df.columns}
    return df.rename(rename_map)

sample_clean = standardize_column_names(sample_df)
sample_clean.head()

### Pandas Comparison

```python
# Pandas: inspect nulls and unique counts
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "nulls": df.isna().sum().values,
    "n_unique": df.nunique(dropna=False).values,
})

# Polars: use expressions and Series methods
profile = pl.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "nulls": [df[col].null_count() for col in df.columns],
    "n_unique": [df[col].n_unique() for col in df.columns],
})
```

Pandas often encourages direct mutation and index-aware operations. In Polars, prefer explicit expressions and column names.

## 5. Analytics Patterns with Polars

Your dashboard must answer at least three meaningful questions. Since the exact table names can vary, this section gives reusable patterns you can adapt after discovering the real columns.

Good project questions usually compare one metric across one or more dimensions:

- Which route has the highest number of flights?
- Which airline has the highest average delay?
- How does traffic change by month or weekday?
- Which airports have the most cancellations?
- What aircraft types are most common?

Start with simple counts, then add rates, averages, and time trends.

In [ ]:
def top_categories(df: pl.DataFrame, category_col: str, top_n: int = 10) -> pl.DataFrame:
    return (
        df
        .lazy()
        .group_by(category_col)
        .agg(pl.len().alias("n_rows"))
        .sort("n_rows", descending=True)
        .head(top_n)
        .collect()
    )

# Example: choose a text-like column from the sampled table.
text_columns = [name for name, dtype in sample_clean.schema.items() if dtype == pl.String]
text_columns

In [ ]:
if text_columns:
    top_categories(sample_clean, text_columns[0], top_n=10)
else:
    print("No text columns found in the selected sample table. Choose another table or use numeric/time analysis.")

In [ ]:
def numeric_summary_by_category(
    df: pl.DataFrame,
    category_col: str,
    value_col: str,
    top_n: int = 10,
) -> pl.DataFrame:
    return (
        df
        .lazy()
        .group_by(category_col)
        .agg(
            pl.len().alias("n_rows"),
            pl.col(value_col).mean().alias(f"avg_{value_col}"),
            pl.col(value_col).median().alias(f"median_{value_col}"),
            pl.col(value_col).min().alias(f"min_{value_col}"),
            pl.col(value_col).max().alias(f"max_{value_col}"),
        )
        .sort(f"avg_{value_col}", descending=True)
        .head(top_n)
        .collect()
    )

numeric_columns = [
    name for name, dtype in sample_clean.schema.items()
    if dtype.is_numeric()
]

print("Text columns:", text_columns)
print("Numeric columns:", numeric_columns)

if text_columns and numeric_columns:
    numeric_summary_by_category(sample_clean, text_columns[0], numeric_columns[0], top_n=10)
else:
    print("Need at least one text column and one numeric column for this pattern.")

In [ ]:
def add_date_parts(df: pl.DataFrame, date_col: str) -> pl.DataFrame:
    """Add common date features after ensuring the selected column is a Polars Date/Datetime."""
    return df.with_columns(
        pl.col(date_col).cast(pl.Date, strict=False).alias(date_col)
    ).with_columns(
        pl.col(date_col).dt.year().alias("year"),
        pl.col(date_col).dt.month().alias("month"),
        pl.col(date_col).dt.weekday().alias("weekday"),
    )


def monthly_trend(df: pl.DataFrame, date_col: str) -> pl.DataFrame:
    df_dates = add_date_parts(df, date_col)
    return (
        df_dates
        .lazy()
        .group_by("year", "month")
        .agg(pl.len().alias("n_rows"))
        .sort("year", "month")
        .collect()
    )

# After loading your real data, use:
# trend = monthly_trend(df, "flight_date")
# trend

### Pandas Comparison

```python
# Pandas
result = (
    df.groupby("airline")
      .agg(n_rows=("airline", "size"), avg_delay=("delay_min", "mean"))
      .sort_values("avg_delay", ascending=False)
      .head(10)
      .reset_index()
)

# Polars
result = (
    df.lazy()
      .group_by("airline")
      .agg(
          pl.len().alias("n_rows"),
          pl.col("delay_min").mean().alias("avg_delay"),
      )
      .sort("avg_delay", descending=True)
      .head(10)
      .collect()
)
```

The key Polars idea is that expressions such as `pl.col("delay_min").mean()` describe work. With `.lazy()`, Polars can optimize that work before running it.

## 6. Prototype Charts in the Notebook

Build charts in the notebook before moving them into Streamlit. Plotly Express accepts Polars DataFrames directly in recent versions.

In [ ]:
if text_columns:
    chart_data = top_categories(sample_clean, text_columns[0], top_n=10)
    fig = px.bar(
        chart_data,
        x=text_columns[0],
        y="n_rows",
        title=f"Top values in {text_columns[0]}",
        labels={text_columns[0]: text_columns[0].replace("_", " ").title(), "n_rows": "Rows"},
    )
    fig.show()
else:
    print("Choose a table with a categorical/text column to create this chart.")

## 7. Export Prepared Data for Streamlit

Reading from the database on every Streamlit rerun can be slow and can put unnecessary load on the class database.

Recommended workflow:

1. Use this notebook to discover and prepare the data.
2. Save cleaned tables or aggregated datasets as Parquet.
3. Make Streamlit read the Parquet files with `pl.read_parquet()`.
4. Only connect to DB2 directly from Streamlit if your group has a specific reason.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Demo export: make the Streamlit template runnable immediately.
# For the final project, replace this with your selected joined/cleaned dataset.
df_demo = read_table(SCHEMA_NAME, TABLE_NAME)
df_clean = standardize_column_names(df_demo)
df_clean.write_parquet(DATA_DIR / "main_clean.parquet")

print(f"Saved {df_clean.height:,} rows and {df_clean.width:,} columns to {DATA_DIR / 'main_clean.parquet'}")

# Example for an aggregated result:
# result_1.write_parquet(DATA_DIR / "top_routes.parquet")

## 8. Create a Simple Streamlit Dashboard

A practical project structure is:

```text
group_project/
├── app.py              # Streamlit UI
├── analysis.py         # Polars transformations and aggregations
├── data/
│   ├── main_clean.parquet
│   └── top_routes.parquet
└── requirements.txt
```

For this course, a single `app.py` is acceptable if the code stays readable. Split into `analysis.py` when your app grows.

In [ ]:
app_code = """
import polars as pl
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="ATT Plane Analytics", layout="wide")

DATA_PATH = "data/main_clean.parquet"


@st.cache_data
def load_data() -> pl.DataFrame:
    return pl.read_parquet(DATA_PATH)


def top_categories(df: pl.DataFrame, category_col: str, top_n: int = 10) -> pl.DataFrame:
    return (
        df
        .lazy()
        .group_by(category_col)
        .agg(pl.len().alias("n_rows"))
        .sort("n_rows", descending=True)
        .head(top_n)
        .collect()
    )


def filter_data(df: pl.DataFrame, selected_value: str | None, category_col: str | None) -> pl.DataFrame:
    if not selected_value or not category_col or selected_value == "All":
        return df

    return (
        df
        .lazy()
        .filter(pl.col(category_col) == selected_value)
        .collect()
    )


st.title("ATT Plane Analytics")

df = load_data()

st.sidebar.header("Filters")
text_columns = [name for name, dtype in df.schema.items() if dtype == pl.String]

if not text_columns:
    st.warning("No text columns found. Add filters manually based on your dataset schema.")
    st.dataframe(df.head(50))
    st.stop()

filter_col = st.sidebar.selectbox("Filter column", text_columns)
values = ["All"] + df.select(pl.col(filter_col).drop_nulls().unique().sort()).to_series().to_list()
selected_value = st.sidebar.selectbox("Value", values)

top_n = st.sidebar.slider("Top N", min_value=5, max_value=30, value=10)

filtered = filter_data(df, selected_value, filter_col)

metric_1, metric_2, metric_3 = st.columns(3)
metric_1.metric("Rows", f"{filtered.height:,}")
metric_2.metric("Columns", f"{filtered.width:,}")
metric_3.metric("Filter", selected_value)

st.subheader(f"Top {top_n} values by selected dimension")
category_col = st.selectbox("Chart dimension", text_columns, index=0)
chart_data = top_categories(filtered, category_col, top_n=top_n)

fig = px.bar(
    chart_data,
    x=category_col,
    y="n_rows",
    title=f"Top {top_n} values in {category_col}",
    labels={category_col: category_col.replace("_", " ").title(), "n_rows": "Rows"},
)
st.plotly_chart(fig, width="stretch")

st.subheader("Data preview")
st.dataframe(filtered.head(100), width="stretch")
"""

Path("app.py").write_text(app_code.strip() + "\n")
print("Wrote app.py template in the current working directory")
print("Run with: uv run streamlit run app.py")

### How to Run the Dashboard

From the repository root:

```bash
uv run streamlit run app.py
```

If your notebook is running from `class_material/s04_group_project`, either move `app.py` into your project folder or run Streamlit from the folder where `app.py` and `data/main_clean.parquet` exist.

### Common Streamlit + Polars Mistakes

| Problem | Cause | Fix |
|---|---|---|
| App is slow on every click | Reading DB2 or recomputing heavy joins every rerun | Cache loads with `@st.cache_data`; save prepared Parquet |
| `LazyFrame` shown instead of data | Forgot `.collect()` | Call `.collect()` before `st.dataframe()` or Plotly |
| Chart order changes | `group_by` order is not guaranteed | Always `.sort(...)` before charting |
| Column not found after join | Duplicate column names got suffixed | Rename before join or use `suffix="_right"` intentionally |

## 9. Group Assignment Requirements

Your final project must include:

- A working Streamlit dashboard.
- Data loaded from the DB2 Plane DB at least once during preparation.
- Polars transformations for cleaning, joining, filtering, and aggregating.
- At least three analysis sections, each answering a clear question.
- At least two interactive filters.
- At least three visualizations.
- A short presentation explaining your most important insights.

Recommended minimum files:

```text
group_<N>_planedb_dashboard/
├── app.py
├── analysis.py                  # optional but recommended
├── data/
│   └── *.parquet                # prepared datasets, if not too large
├── requirements.txt
└── README.md                    # how to run + key findings
```

## 10. Work Plan for Today

Use the remaining time in session 4 to complete these steps:

1. Confirm your group account can connect.
2. List the tables and choose the relevant ones.
3. Read small samples and document schemas.
4. Identify at least three business questions.
5. Build one working Polars aggregation per question.
6. Export prepared data to Parquet.
7. Start the Streamlit dashboard with one filter and one chart.

Bring to session 5: a dashboard that runs, even if the design and final insights are still rough.